# Laboratorio 4 — Cuaderno 2: el índice de cianobacteria, NDVI y NDWI

**Ejercicio 3.** Aplicación del script oficial de detección de cianobacteria de Sentinel Hub,
más los índices NDVI y NDWI, a cada una de las 22 escenas.

## De qué se trata

Las cianobacterias son microorganismos que viven en el agua y que, cuando encuentran mucho
nutriente y temperatura alta, se multiplican hasta formar manchas verdes visibles desde el
espacio. Esas manchas se llaman *floraciones* y algunas son tóxicas.

No podemos ver la cianobacteria directamente desde un satélite, pero sí podemos ver el
pigmento que usa para hacer fotosíntesis: la **clorofila-a**. Ese pigmento tiene una firma
óptica muy particular —absorbe luz roja y refleja en una banda estrecha justo después del
rojo, el llamado *borde rojo*— y Sentinel-2 tiene una banda puesta exactamente ahí (B05,
705 nm). Comparar cuánta luz vuelve en el rojo contra cuánta vuelve en el borde rojo es lo que
permite estimar cuánta clorofila hay, y con ella cuánta floración.

## 1. El script que usamos

Usamos el script **CyanoLakes Chlorophyll-a**, de Jeremy Kravitz y Mark Matthews (2020),
publicado en el repositorio oficial de scripts de Sentinel Hub:

<https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/cyanobacteria_chla_ndci_l1c/>

El script original está escrito en JavaScript y corre dentro de Sentinel Hub, píxel por píxel,
devolviendo un **color**. Para este laboratorio lo tradujimos a Python de forma vectorizada, sin
cambiar ninguna fórmula ni ningún umbral, por dos razones prácticas:

1. Necesitamos el **valor numérico** del índice, no solo el color, para poder promediarlo,
   graficarlo en el tiempo y correlacionarlo con NDVI y NDWI.
2. Así el cálculo queda reproducible en el repositorio y no depende de una sesión del navegador.

La traducción está en `src/indices.py`. El script trabaja en cuatro pasos.

### Paso 1 — Encontrar dónde hay agua

Antes de medir clorofila hay que saber qué píxeles son lago y cuáles son tierra: un bosque
también refleja fuerte en el borde rojo y daría un valor altísimo y falso.

El script combina seis criterios distintos de detección de agua y basta con que uno se cumpla:

| Criterio | Se cumple si | Qué detecta |
|----------|--------------|-------------|
| MNDWI | > 0.42 | Agua, usando el contraste verde/SWIR |
| NDWI | > 0.40 | Agua, usando el contraste verde/infrarrojo |
| AWEI sombra | > 0.1879 | Agua en zonas sombreadas |
| AWEI sin sombra | > 0.1112 | Agua en terreno abierto |
| NDVI | < −0.2 | Superficies sin nada de vegetación |
| NDWI hojas | > 1 | Vegetación acuática flotante |

Después aplica un filtro de descarte: si el píxel se parece a zona urbana o a suelo desnudo
—que ópticamente pueden confundirse con agua— se le quita la etiqueta de agua.

Este paso es importante para Amatitlán, que tiene la ciudad de Guatemala pegada al norte.

### Paso 2 — Separar la nata flotante

Cuando la floración es tan densa que forma una capa en la superficie, deja de comportarse como
agua con algas disueltas y pasa a comportarse como vegetación flotante. El modelo de clorofila
ya no aplica ahí porque se satura.

El script usa el **FAI** (índice de algas flotantes), que traza una línea recta entre lo que se
ve a 665 nm y a 865 nm y mide cuánto se sale de esa línea la lectura de 783 nm. Si el exceso
supera 0.08, el píxel se marca como material flotante y se pinta de rojo-naranja.

### Paso 3 — Estimar la clorofila

El índice base es el **NDCI** (índice normalizado de diferencia de clorofila):

$$\text{NDCI} = \frac{B05 - B04}{B05 + B04} = \frac{\text{borde rojo} - \text{rojo}}{\text{borde rojo} + \text{rojo}}$$

Entre más clorofila hay, más absorbe el rojo y más refleja el borde rojo, así que el NDCI sube.
Luego el script convierte NDCI a concentración con un polinomio calibrado:

$$\text{Chl-a} = 826.57 \cdot \text{NDCI}^3 - 176.43 \cdot \text{NDCI}^2 + 19 \cdot \text{NDCI} + 4.071$$

El resultado está en **microgramos por litro (µg/L)**, que es la unidad con la que se reportan
los análisis de laboratorio de calidad de agua. Eso permite comparar contra umbrales sanitarios
reconocidos en vez de contra un número abstracto.

### Paso 4 — Pintar el resultado

El script asigna un color según la concentración, con una escala de 26 tramos que va del azul
(agua limpia) al verde y al naranja-rojo (floración severa). Reproducimos esa misma escala para
que nuestros mapas se vean igual que en Copernicus Browser.

### Dos controles de calidad que agregamos nosotros

El script original supone que recibe píxeles limpios. Como aquí lo corremos sobre escenas
completas, agregamos dos filtros propios —que **no alteran ninguna fórmula del script**, solo
deciden a qué píxeles tiene sentido aplicárselo.

**1. Descarte de nubes.** L1C no trae la capa de clasificación de escena que sí tiene L2A, así
que la nube se detecta con dos pruebas sobre la propia reflectancia: valor apreciable en B10
(que solo puede venir de nube alta, porque el vapor de agua bloquea esa longitud de onda desde
el suelo) y brillo simultáneo en azul y verde, que delata nube densa. De todos modos la nube
casi nunca sobrevive a la máscara de agua, porque una nube no cumple ningún criterio de agua.

**2. Piso de señal.** Si el rojo y el borde rojo llegan prácticamente en cero, su cociente
normalizado es ruido dividido entre ruido. Exigimos una reflectancia mínima de 0.005 en ambas
bandas y marcamos como no válido lo que no llegue. Es preferible declarar que ahí no se puede
medir a reportar un número inventado.

Este segundo filtro es el que hizo evidente que L2A no servía para este script: sobre Atitlán
descartaba más de la mitad del lago. Con L1C no descarta prácticamente nada.

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, datos, graficos
from src import indices as ix

escenas = datos.escenas_disponibles()
print(f"Escenas disponibles: {len(escenas)} de 22")
if not escenas:
    raise SystemExit("Corre primero: .venv/bin/python -m src.descarga")

## 2. El cálculo, paso a paso, sobre una escena

Tomamos una fecha de cada lago y seguimos el encadenamiento completo para ver qué hace cada
etapa.

In [ ]:
def mostrar_pasos(lago, fecha):
    capas, meta = datos.indices_escena(lago, fecha)
    fig, ejes = plt.subplots(1, 4, figsize=(19, 5))

    ejes[0].imshow(graficos.color_verdadero_realzado(capas))
    ejes[0].set_title("1. Color verdadero\n(lo que vería el ojo)")

    ejes[1].imshow(capas["agua"], cmap="Blues", interpolation="nearest")
    n_agua = capas["agua"].sum()
    ejes[1].set_title(f"2. Máscara de agua\n{n_agua:,} píxeles = "
                      f"{n_agua * datos.AREA_PIXEL_KM2:.1f} km²")

    ndci_agua = np.where(capas["agua_valida"], capas["ndci"], np.nan)
    im = ejes[2].imshow(ndci_agua, cmap="RdYlGn", vmin=-0.1, vmax=0.4,
                        interpolation="nearest")
    ejes[2].set_title("3. NDCI\n(clorofila cruda)")
    plt.colorbar(im, ax=ejes[2], fraction=0.046)

    graficos.dibujar_mapa_chl(ejes[3], capas, "4. Clorofila-a\n(escala oficial)")
    graficos.barra_color_chl(fig, ejes[3])

    for eje in ejes:
        eje.set_xticks([]); eje.set_yticks([])

    fig.suptitle(f"{config.NOMBRE_LARGO[lago]} — {fecha}", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

    valores = capas["chl_agua"][capas["agua_valida"]]
    print(f"Clorofila-a en el agua: media {valores.mean():.1f} µg/L | "
          f"mediana {np.median(valores):.1f} | máximo {valores.max():.1f}")
    print(f"Cobertura válida (sin nubes): "
          f"{100 * capas['agua_valida'].sum() / capas['agua'].sum():.1f}% del espejo de agua")

mostrar_pasos("Atitlan", config.FECHAS["Atitlan"][0])

In [ ]:
mostrar_pasos("Amatitlan", config.FECHAS["Amatitlan"][0])

## 3. Índice de cianobacteria en todas las fechas

Ahora aplicamos el script a las 11 fechas de cada lago. Estos son los mapas que piden los
ejercicios 3 y 5.2: la distribución de cianobacteria dentro de cada lago, y la comparación
entre fechas.

In [ ]:
def rejilla_chl(lago, columnas=4):
    fechas = [f for f in config.FECHAS[lago] if config.ruta_tif(lago, f).exists()]
    filas = int(np.ceil(len(fechas) / columnas))
    fig, ejes = plt.subplots(filas, columnas, figsize=(4.2 * columnas, 4.4 * filas))
    ejes = np.atleast_1d(ejes).ravel()

    for eje, fecha in zip(ejes, fechas):
        capas, _ = datos.indices_escena(lago, fecha)
        valores = capas["chl_agua"][capas["agua_valida"]]
        media = valores.mean() if valores.size else np.nan
        graficos.dibujar_mapa_chl(eje, capas, f"{fecha}\nmedia {media:.1f} µg/L")

    for eje in ejes[len(fechas):]:
        eje.axis("off")

    graficos.barra_color_chl(fig, list(ejes[:len(fechas)]))
    fig.suptitle(
        f"Índice de cianobacteria (clorofila-a) — {config.NOMBRE_LARGO[lago]}",
        fontsize=15, y=1.0,
    )
    fig.tight_layout()
    plt.show()

rejilla_chl("Atitlan")

In [ ]:
rejilla_chl("Amatitlan")

### Cómo leer estos mapas

| Color | Clorofila-a | Estado del agua |
|-------|-------------|-----------------|
| Azul intenso | < 2.5 µg/L | Agua limpia, oligotrófica |
| Azul claro / turquesa | 2.5 – 7 µg/L | Mesotrófica, productividad moderada |
| Verde | 7 – 30 µg/L | Eutrófica: hay floración en curso |
| Amarillo | 30 – 90 µg/L | Floración intensa |
| Naranja | 90 – 450 µg/L | Floración severa |
| Rojo-naranja | > 500 µg/L o nata flotante | Floración extrema en superficie |

Como referencia sanitaria, la Organización Mundial de la Salud ubica alrededor de **50 µg/L**
de clorofila-a el nivel donde el riesgo para uso recreativo deja de ser moderado. Ese es el
umbral que usamos más adelante para medir qué tan extendida está una floración.

## 4. NDVI y NDWI

Los otros dos índices que pide el laboratorio. Ambos se calculan localmente con las bandas que
ya descargamos.

- **NDVI** = (B08 − B04) / (B08 + B04). Mide vigor de vegetación. Sobre tierra indica qué tan
  densa y sana está la cobertura vegetal; sobre agua, valores que suben delatan material
  vegetal en la superficie.
- **NDWI** = (B03 − B08) / (B03 + B08). Mide presencia de agua. Sirve para delimitar el espejo
  de agua y para ver cambios en su extensión.

In [ ]:
def mapa_ndvi_ndwi(lago, fecha):
    capas, _ = datos.indices_escena(lago, fecha)
    fig, ejes = plt.subplots(1, 3, figsize=(16, 5))

    ejes[0].imshow(graficos.color_verdadero_realzado(capas))
    ejes[0].set_title("Color verdadero")

    im1 = ejes[1].imshow(capas["ndvi"], cmap="YlGn", vmin=-0.3, vmax=0.9,
                         interpolation="nearest")
    ejes[1].set_title("NDVI")
    plt.colorbar(im1, ax=ejes[1], fraction=0.046, label="NDVI")

    im2 = ejes[2].imshow(capas["ndwi"], cmap="BrBG", vmin=-1, vmax=1,
                         interpolation="nearest")
    ejes[2].set_title("NDWI")
    plt.colorbar(im2, ax=ejes[2], fraction=0.046, label="NDWI")

    for eje in ejes:
        eje.set_xticks([]); eje.set_yticks([])

    fig.suptitle(f"{config.NOMBRE_LARGO[lago]} — {fecha}", fontsize=14)
    fig.tight_layout()
    plt.show()

for lago in ["Atitlan", "Amatitlan"]:
    mapa_ndvi_ndwi(lago, config.FECHAS[lago][0])

| NDVI | Interpretación |
|------|----------------|
| 0.6 a 1.0 | Vegetación muy densa y sana |
| 0.4 a 0.6 | Vegetación saludable (cultivos, bosque) |
| 0.2 a 0.4 | Vegetación escasa o en estrés |
| 0.0 a 0.2 | Suelo desnudo, hierba seca |
| < 0.0 | Agua, nubes o superficies artificiales |

| NDWI | Interpretación |
|------|----------------|
| > 0.2 | Agua o zonas húmedas |
| 0 a 0.2 | Vegetación poco densa o suelo húmedo |
| < 0 | Vegetación o suelo seco |

En ambos mapas el lago aparece como una silueta muy nítida: NDVI marcadamente negativo y NDWI
marcadamente positivo. Ese contraste es lo que hace confiable la delimitación automática del
espejo de agua.

## 5. Tabla resumen de las 22 escenas

Consolidamos todo en una tabla con una fila por escena. Es la base de los cuadernos siguientes
y queda guardada en `data/derived/resumen_escenas.csv`.

In [ ]:
resumen = datos.tabla_resumen(recalcular=True)

vista = resumen.copy()
vista["fecha"] = vista["fecha"].dt.strftime("%Y-%m-%d")
columnas = ["lago", "fecha", "area_agua_km2", "cobertura_valida_pct",
            "chl_media", "chl_mediana", "chl_p90", "chl_max",
            "pct_alto", "pct_flotante", "ndvi_media", "ndwi_media", "nubosidad_pct"]
vista[columnas].round(2)

### Control de calidad antes de seguir

Dos cosas que conviene verificar: que la máscara de agua da un área estable y creíble en todas
las fechas, y que ninguna escena quedó demasiado tapada por nubes.

In [ ]:
for lago in ["Atitlan", "Amatitlan"]:
    sub = resumen[resumen["lago"] == lago]
    print(f"\n{config.NOMBRE_LARGO[lago]}")
    print(f"  Área de agua detectada: {sub['area_agua_km2'].min():.1f} – "
          f"{sub['area_agua_km2'].max():.1f} km² "
          f"(mediana {sub['area_agua_km2'].median():.1f})")
    print(f"  Cobertura válida: {sub['cobertura_valida_pct'].min():.1f}% – "
          f"{sub['cobertura_valida_pct'].max():.1f}%")
    flojas = sub[sub["cobertura_valida_pct"] < 70]
    if len(flojas):
        print("  Escenas con menos del 70% de cobertura válida:")
        for _, f in flojas.iterrows():
            print(f"    - {f['fecha']:%Y-%m-%d}: {f['cobertura_valida_pct']:.1f}%")
    else:
        print("  Todas las escenas superan el 70% de cobertura válida.")

El área superficial publicada de Atitlán ronda los **130 km²** y la de Amatitlán los **15 km²**.
Si los valores detectados quedan en ese orden, la máscara de agua está funcionando bien.

---

**Siguiente:** `03_Analisis_Temporal.ipynb` — evolución del índice a lo largo del tiempo.